# Lab 8: Predictive Analysis (Machine Learning)

**Goal:** Train several classification models to predict Falcon 9 first-stage landing success (`Class`), tune hyperparameters, evaluate each on a held-out test set, and pick the best-performing model with a confusion matrix and justification.

Models compared: **Logistic Regression, Support Vector Machine, Decision Tree, K-Nearest Neighbors.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, jaccard_score, f1_score, classification_report

sns.set_style('whitegrid')
%matplotlib inline

## 0. Load the target (`Class`) and the one-hot encoded feature matrix
`dataset_part_2.csv` (Lab 3) has the target `Class`. `dataset_part_3.csv` (Lab 4) has the one-hot encoded numeric features.

In [ ]:
def load_or_fallback(local_name, fallback_url):
    if os.path.exists(local_name):
        return pd.read_csv(local_name)
    print(f'{local_name} not found locally, loading from IBM dataset repository...')
    return pd.read_csv(fallback_url)

data = load_or_fallback(
    'dataset_part_2.csv',
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
)
X = load_or_fallback(
    'dataset_part_3.csv',
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv'
)

Y = data['Class'].to_numpy()
print('Features shape:', X.shape, '| Target shape:', Y.shape)
X.head()

## 1. Standardize features and split into train/test sets

In [ ]:
X = X.astype(float)
transform = preprocessing.StandardScaler()
X_scaled = transform.fit_transform(X)

X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=2)
print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])

## 2. Helper to plot a confusion matrix

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Failure', 'Success'], yticklabels=['Failure', 'Success'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(title)
    plt.show()

## 3. Logistic Regression (with GridSearchCV)

In [ ]:
parameters_lr = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr = LogisticRegression(max_iter=1000)
logreg_cv = GridSearchCV(lr, parameters_lr, cv=10)
logreg_cv.fit(X_train, Y_train)

print('Best params:', logreg_cv.best_params_)
print('Best CV accuracy:', logreg_cv.best_score_)

lr_test_acc = logreg_cv.score(X_test, Y_test)
print('Test accuracy:', lr_test_acc)

yhat_lr = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_lr, 'Logistic Regression')

## 4. Support Vector Machine (with GridSearchCV)

In [ ]:
parameters_svm = {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
                   'C': np.logspace(-3, 3, 5),
                   'gamma': np.logspace(-3, 3, 5)}
svm = SVC()
svm_cv = GridSearchCV(svm, parameters_svm, cv=10)
svm_cv.fit(X_train, Y_train)

print('Best params:', svm_cv.best_params_)
print('Best CV accuracy:', svm_cv.best_score_)

svm_test_acc = svm_cv.score(X_test, Y_test)
print('Test accuracy:', svm_test_acc)

yhat_svm = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_svm, 'Support Vector Machine')

## 5. Decision Tree (with GridSearchCV)

In [ ]:
parameters_tree = {'criterion': ['gini', 'entropy'],
                    'splitter': ['best', 'random'],
                    'max_depth': [2*n for n in range(1, 10)],
                    'max_features': ['sqrt'],
                    'min_samples_leaf': [1, 2, 4],
                    'min_samples_split': [2, 5, 10]}
tree = DecisionTreeClassifier(random_state=2)
tree_cv = GridSearchCV(tree, parameters_tree, cv=10)
tree_cv.fit(X_train, Y_train)

print('Best params:', tree_cv.best_params_)
print('Best CV accuracy:', tree_cv.best_score_)

tree_test_acc = tree_cv.score(X_test, Y_test)
print('Test accuracy:', tree_test_acc)

yhat_tree = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_tree, 'Decision Tree')

## 6. K-Nearest Neighbors (with GridSearchCV)

In [ ]:
parameters_knn = {'n_neighbors': list(range(1, 11)),
                   'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
                   'p': [1, 2]}
knn = KNeighborsClassifier()
knn_cv = GridSearchCV(knn, parameters_knn, cv=10)
knn_cv.fit(X_train, Y_train)

print('Best params:', knn_cv.best_params_)
print('Best CV accuracy:', knn_cv.best_score_)

knn_test_acc = knn_cv.score(X_test, Y_test)
print('Test accuracy:', knn_test_acc)

yhat_knn = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat_knn, 'K-Nearest Neighbors')

## 7. Compare all models and pick the best one

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Decision Tree', 'KNN'],
    'Best CV Accuracy': [logreg_cv.best_score_, svm_cv.best_score_, tree_cv.best_score_, knn_cv.best_score_],
    'Test Accuracy': [lr_test_acc, svm_test_acc, tree_test_acc, knn_test_acc]
}).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)

results

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=results, x='Model', y='Test Accuracy', palette='viridis')
plt.ylim(0, 1)
plt.title('Model Comparison — Test Accuracy')
plt.ylabel('Test Accuracy')
plt.show()

best_model_name = results.iloc[0]['Model']
print(f"Best performing model: {best_model_name}")
print(results.to_string(index=False))

## 8. Write-up for your slide (1.15)
Fill this in with your actual printed numbers once you've run the notebook:

- **Best model:** *(fill in from the table above)*
- **Why it won:** compare test accuracy and how well it balances false positives (predicting success when it failed) vs. false negatives (predicting failure when it succeeded) using the confusion matrix.
- **Conclusion:** summarize what features seemed to drive success (recall your EDA — flight number, orbit type, payload mass) and what this means for predicting future launch outcomes.